<a href="https://colab.research.google.com/github/robertbarcik/vector_databases-tutorial/blob/main/1.%20ChromaDB/1_Creating_Embeddings_using_Chroma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Embeddings and a Real Vector Store with ChromaDB

In the previous course you built RAG by hand: ten company profiles, one embedding call per document, a pandas table with a `vector` column, and a cosine loop to find the nearest neighbours. That table *was* a vector database, only without the persistence, speed and filtering you need once there are more than a few hundred documents.

This notebook swaps the pandas table for **ChromaDB**, an open-source vector database you can run inside a notebook with no server and no account. The idea stays exactly the same: text in, numbers out, numbers stored, nearest neighbours found. What you gain is a proper store: it survives a restart, it indexes millions of vectors, and it filters by metadata.

Two notebooks make up this module. Here we **create the store**: embeddings, collections, metadata, updates. In the next one we **use it**: semantic search and a RAG pipeline on top of it.


## Setup

Everything runs locally except one optional section at the end that calls OpenAI's embedding model. The key lookup goes Colab secret, then environment variable, then a prompt. The six sample documents are fetched from the course repository when they are not next to the notebook (Colab).


In [1]:
%pip install -q chromadb==1.5.5 openai==2.28.0   # Colab installs here; locally, `pip install -r requirements.txt` in the course folder already covers it

import os, json, pathlib, urllib.request

# Your OpenAI key (only needed for section 6). Colab secret -> environment variable -> a prompt.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key (press Enter to skip section 6): ")
os.environ["OPENAI_API_KEY"] = api_key or ""

# Data file: next to this notebook if you downloaded the course folder; fetched from GitHub otherwise (Colab).
RAW = "https://raw.githubusercontent.com/robertbarcik/vector_databases-tutorial/main/1.%20ChromaDB/"
if not pathlib.Path("cyber_documents.json").exists():
    urllib.request.urlretrieve(RAW + "cyber_documents.json", "cyber_documents.json")

os.environ["ANONYMIZED_TELEMETRY"] = "False"   # keep Chroma's usage telemetry quiet
import chromadb
from chromadb.utils import embedding_functions

print("Ready. chromadb", chromadb.__version__)


Note: you may need to restart the kernel to use updated packages.


Ready. chromadb 1.5.5


## 1. One word becomes 384 numbers

An **embedding function** is the piece of code that turns text into a vector. Chroma ships with a default one: the small open model `all-MiniLM-L6-v2`, downloaded once (about 80 MB) and run on your CPU. No API, no cost, no data leaving your machine. Let's embed a single word and look at what comes back.


In [2]:
embed_local = embedding_functions.DefaultEmbeddingFunction()

vectors = embed_local(["Robert"])

print("texts in:", 1, "| vectors out:", len(vectors))
print("numbers per vector:", len(vectors[0]))
print("the first five:", [round(float(x), 3) for x in vectors[0][:5]])


texts in: 1 | vectors out: 1
numbers per vector: 384
the first five: [-0.054, 0.027, -0.107, -0.004, 0.025]


### 🔍 What just happened?

One word went in, 384 numbers came out. Nobody chose what each number means: the model learned during training to place texts with similar meaning close together in this 384-dimensional space. The previous course used OpenAI's `text-embedding-3-small` with 1,536 numbers; the local model is smaller and free. Both do the same job.

### 🎯 Mini-task

Embed a list of three to five words from one topic you like (programming languages, foods, cities). Print how many vectors came back and how long each one is.


In [3]:
# YOUR CODE HERE


## 2. A place to keep them: client and collection

Embedding one word is easy. The real job is keeping thousands of vectors and finding the right ones later. Chroma organises this in two layers:

- a **client** is the database itself. `chromadb.EphemeralClient()` keeps everything in memory and forgets it when the notebook stops; `chromadb.PersistentClient(path=...)` writes to a folder on disk so you can come back tomorrow.
- a **collection** is a table inside the database: one embedding function, one distance metric, many documents.

We will use a persistent client, so the store you build here can be reopened in the next notebook.


In [4]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(
    name="cyber_docs",
    embedding_function=embed_local,
    configuration={"hnsw": {"space": "cosine"}},   # compare vectors by cosine distance
)

print("collections in this database:", [c.name for c in client.list_collections()])
print("documents in 'cyber_docs':", collection.count())


collections in this database: ['cyber_docs']
documents in 'cyber_docs': 0


### 🔍 What just happened?

A folder called `chroma_db` appeared next to the notebook, and inside the database there is now one empty collection. Two decisions were made here that you cannot change later without recreating the collection: **which embedding function** turns text into vectors (the local one) and **which distance** compares them (cosine, the same measure you computed by hand in the previous course). `get_or_create_collection` is safe to re-run: it returns the existing collection instead of failing.


## 3. Adding documents

Six short articles about cybersecurity, each with a title, a text and a few **metadata** fields (category, year, difficulty, author). Let's look at one before we store them.


In [5]:
documents = json.load(open("cyber_documents.json"))

print(len(documents), "documents\n")
for d in documents:
    print(f"{d['id']:<11} {d['title']:<30} {d['metadata']}")

print("\nThe start of the first one:\n", documents[0]["text"][:300], "...")


6 documents

document_1  Public-key cryptography        {'category': 'cryptography', 'year': 1976, 'difficulty': 'advanced', 'author': 'Security Team'}
document_2  Stuxnet                        {'category': 'cyberwarfare', 'year': 2010, 'difficulty': 'intermediate', 'author': 'Security Team'}
document_3  Zero Trust Architecture        {'category': 'architecture', 'year': 2010, 'difficulty': 'intermediate', 'author': 'Security Team'}
document_4  Phishing                       {'category': 'social-engineering', 'year': 1995, 'difficulty': 'beginner', 'author': 'Security Team'}
document_5  Multi-Factor Authentication    {'category': 'authentication', 'year': 2020, 'difficulty': 'beginner', 'author': 'Security Team'}
document_6  Virtual Private Networks       {'category': 'network-security', 'year': 2015, 'difficulty': 'beginner', 'author': 'Network Team'}

The start of the first one:
 Public-key cryptography, also known as asymmetric cryptography, represents a monumental paradigm shift f

`collection.add()` takes parallel lists: one id, one text and one metadata dictionary per document. Chroma runs the embedding function on every text for you and builds the index. The text is embedded; the metadata is stored next to it, not embedded, so you can filter on it later.


In [6]:
collection.add(
    ids=[d["id"] for d in documents],
    documents=[d["text"] for d in documents],
    metadatas=[d["metadata"] for d in documents],
)

print("documents in the collection:", collection.count())


documents in the collection: 6


### 🔍 What just happened?

Six texts went through the embedding model and six rows landed in the collection, each with its id, its text, its metadata and its 384-number vector. Let's pull one row back out and look at all four parts.


In [7]:
row = collection.get(ids=["document_2"], include=["documents", "metadatas", "embeddings"])

print("id:      ", row["ids"][0])
print("metadata:", row["metadatas"][0])
print("text:    ", row["documents"][0][:120], "...")
print("vector:  ", [round(float(x), 3) for x in row["embeddings"][0][:6]], "... (", len(row["embeddings"][0]), "numbers )")


id:       document_2
metadata: {'author': 'Security Team', 'year': 2010, 'difficulty': 'intermediate', 'category': 'cyberwarfare'}
text:     Stuxnet stands as a watershed moment in the history of digital warfare, a malicious computer worm that transcended the d ...
vector:   [-0.079, 0.01, -0.029, -0.011, 0.047, -0.03] ... ( 384 numbers )


### 🎯 Mini-task

Add a seventh document of your own with `collection.add()`: a few sentences about firewalls, ransomware or anything security-related, id `document_7`, with the same four metadata keys. Check `collection.count()` afterwards. If you run the cell twice, Chroma will complain that the id already exists; that is the point of ids.


In [8]:
# YOUR CODE HERE


## 4. Filtering with `where`

Metadata is what makes a vector store usable in a real application: "only this customer's documents", "only contracts from 2025", "only beginner material". The `where` argument takes a dictionary, and it works both for plain retrieval with `get()` and, as you will see in the next notebook, combined with semantic search.


In [9]:
beginner = collection.get(where={"difficulty": "beginner"}, include=["metadatas"])

print("beginner-level documents:")
for doc_id, meta in zip(beginner["ids"], beginner["metadatas"]):
    print(f"  {doc_id}: {meta['category']} ({meta['year']})")


beginner-level documents:
  document_4: social-engineering (1995)
  document_5: authentication (2020)
  document_6: network-security (2015)


Comparisons use MongoDB-style operators: `$gt`, `$gte`, `$lt`, `$lte`, `$ne`, `$in`, and `$and` / `$or` to combine conditions.


In [10]:
recent_by_security_team = collection.get(
    where={"$and": [{"year": {"$gte": 2010}}, {"author": "Security Team"}]},
    include=["metadatas"],
)

print("year >= 2010 AND author = Security Team:")
for doc_id, meta in zip(recent_by_security_team["ids"], recent_by_security_team["metadatas"]):
    print(f"  {doc_id}: {meta['category']} ({meta['year']}, {meta['author']})")


year >= 2010 AND author = Security Team:
  document_2: cyberwarfare (2010, Security Team)
  document_3: architecture (2010, Security Team)
  document_5: authentication (2020, Security Team)


### 🎯 Mini-task

Get every document whose category is in the list `["cryptography", "authentication"]` (hint: `$in`). Print the ids.


In [11]:
# YOUR CODE HERE


## 5. Changing and removing rows

Documents get corrected, re-tagged and retired. `upsert()` updates a row if the id exists and inserts it otherwise; `delete()` removes rows by id or by a `where` filter. Let's mark one document as reviewed, add a temporary note, then remove the note again.


In [12]:
# Update: same id, same text, one extra metadata key.
doc5 = collection.get(ids=["document_5"], include=["documents", "metadatas"])
collection.upsert(
    ids=["document_5"],
    documents=doc5["documents"],
    metadatas=[{**doc5["metadatas"][0], "reviewed": True}],
)
print("document_5 metadata now:", collection.get(ids=["document_5"], include=["metadatas"])["metadatas"][0])

# Insert: a new id.
collection.upsert(
    ids=["note_1"],
    documents=["Reminder: rotate the VPN certificates before the end of the quarter."],
    metadatas=[{"category": "note", "year": 2026, "difficulty": "beginner", "author": "Demo"}],
)
print("count after the note:", collection.count())


document_5 metadata now: {'reviewed': True, 'difficulty': 'beginner', 'year': 2020, 'category': 'authentication', 'author': 'Security Team'}


count after the note: 7


In [13]:
collection.delete(where={"author": "Demo"})   # or: collection.delete(ids=["note_1"])

print("count after deleting the note:", collection.count())
print("ids left:", collection.get()["ids"])


count after deleting the note: 6
ids left: ['document_1', 'document_2', 'document_3', 'document_4', 'document_5', 'document_6']


### 🔍 What just happened?

The reviewed flag was added without re-embedding anything else, the note was inserted and then removed by a metadata filter, and the six original documents are untouched. Two things to notice: `upsert` with a changed **text** re-embeds that document, and `delete` is final, there is no undo.


## 6. The same store with OpenAI embeddings

Nothing in the store depends on the local model. Swap the embedding function and the same code produces a collection of 1,536-number vectors computed on OpenAI's servers, the model you used in the previous course. For six short texts the cost is a fraction of a cent. This section needs the API key; skip it if you did not enter one.


In [14]:
embed_openai = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small",
)

openai_collection = client.get_or_create_collection(
    name="cyber_docs_openai",
    embedding_function=embed_openai,
    configuration={"hnsw": {"space": "cosine"}},
)

if openai_collection.count() == 0:
    openai_collection.add(
        ids=[d["id"] for d in documents],
        documents=[d["text"] for d in documents],
        metadatas=[d["metadata"] for d in documents],
    )

row = openai_collection.get(ids=["document_1"], include=["embeddings"])
print("documents:", openai_collection.count(), "| numbers per vector:", len(row["embeddings"][0]))
print("collections now:", [c.name for c in client.list_collections()])


documents: 6 | numbers per vector: 1536
collections now: ['cyber_docs_openai', 'cyber_docs']


### 🔍 What just happened?

Same six documents, same `add()` call, a different model behind it: 1,536 numbers per document instead of 384. The two collections cannot be mixed, a query embedded with one model means nothing in the other's space, which is why the embedding function is fixed per collection.

Which to choose? The local model is free, private and good enough for many tasks in English. OpenAI's model is stronger, multilingual (it handles Slovak text well) and costs about 0.02 USD per million tokens. In the next notebook we search the local collection; try the OpenAI one as an exercise.


## 7. Coming back tomorrow

Because the client is persistent, the vectors are on disk in `chroma_db/`. A new session only needs to point a client at the same folder and open the collection by name with `get_collection()`, passing the same embedding function so that queries are embedded the same way as the stored documents. No re-embedding, no re-upload.


In [15]:
client_again = chromadb.PersistentClient(path="./chroma_db")
reopened = client_again.get_collection("cyber_docs", embedding_function=embed_local)

print("reopened 'cyber_docs' with", reopened.count(), "documents")
print("categories:", sorted({m["category"] for m in reopened.get(include=["metadatas"])["metadatas"]}))


reopened 'cyber_docs' with 6 documents
categories: ['architecture', 'authentication', 'cryptography', 'cyberwarfare', 'network-security', 'social-engineering']


## What you take with you

- An **embedding function** turns text into a vector; Chroma's default is a free local model, OpenAI's is a paid API, and a collection is tied to one of them.
- A **collection** holds ids, texts, metadata and vectors; `add`, `get`, `upsert`, `delete` are the whole CRUD vocabulary.
- **Metadata filters** (`where`) are how a vector store meets business rules: permissions, dates, document types.
- A **persistent client** keeps everything in a folder, so the next notebook can open this exact store and search it.

Next up: semantic search with `query()`, metadata-filtered search, and a RAG pipeline on top of this collection.
